In [ ]:
import os
import numpy as np
from pathlib import Path
import scipy.ndimage
import SimpleITK as sitk

import scipy.ndimage as ndi

def get_bounding_box(pet_array, ct_array):
    """
    Improved 3D bounding box cropping algorithm, extracts the largest connected component
    to effectively remove background noise and CT bed edges.
    """
    # 1. Rely solely on CT to find the body contour.
    # The HU of skin/muscle is typically between -100 and +100.
    # Using -500 safely captures the outermost skin contour of the body while perfectly 
    # filtering out air (-1000) and mild scatter noise.
    body_mask = ct_array > -500
    
    # 2. Extract Connected Components
    # This step effectively eliminates ring artifacts or scattered external objects 
    # at the edge of the field of view (FOV).
    labels, num_features = ndi.label(body_mask)
    if num_features == 0:
        return None
        
    # 3. Find the largest connected component (ignoring background 0)
    counts = np.bincount(labels.ravel())
    counts[0] = 0  # Set the count of the background to 0
    largest_component_label = counts.argmax()
    
    # Generate a clean body mask
    clean_body_mask = (labels == largest_component_label)
    
    # 4. Get compact coordinate boundaries
    z_indices, y_indices, x_indices = np.where(clean_body_mask)
    
    if len(z_indices) == 0:
        return None
        
    z_min, z_max = z_indices.min(), z_indices.max()
    y_min, y_max = y_indices.min(), y_indices.max()
    x_min, x_max = x_indices.min(), x_indices.max()
    
    # 5. Add a little padding (e.g., 3 voxels to prevent loss of convolutional features due to tight outer contours)
    pad = 3
    z_min, z_max = max(0, z_min - pad), min(pet_array.shape[0], z_max + pad + 1)
    y_min, y_max = max(0, y_min - pad), min(pet_array.shape[1], y_max + pad + 1)
    x_min, x_max = max(0, x_min - pad), min(pet_array.shape[2], x_max + pad + 1)
    
    return slice(z_min, z_max), slice(y_min, y_max), slice(x_min, x_max)

def normalize_volume(volume, method='z-score'):
    """
    Normalizes a 3D volume using either Z-score or Min-Max scaling.
    """
    if method == 'z-score':
        mu = np.mean(volume)
        sigma = np.std(volume)
        if sigma == 0:
            return volume - mu
        return (volume - mu) / sigma
    elif method == 'min-max':
        v_min = np.min(volume)
        v_max = np.max(volume)
        if v_max - v_min == 0:
            return volume - v_min
        return (volume - v_min) / (v_max - v_min)
    else:
        raise ValueError("Method must be 'z-score' or 'min-max'")

def process_autopet_dataset():
    """
    Standalone preprocessing pipeline for the AutoPET dataset.
    Uses recursive globbing to handle nested scan directories.
    """
    # 1. Define source and target directories
    source_base = Path('/path/to/your/dataset/AutoPET2025_FDG')
    target_base = Path('/path/to/your/output/PETCTfoundation/AutoPET')
    
    # Corrected target spacing: (Z=3.0, Y=2.0, X=2.0)
    target_spacing_zyx = np.array([3.0, 2.0, 2.0])
    
    print(f"Starting AutoPET preprocessing...")
    print(f"Source: {source_base}")
    print(f"Target: {target_base}\n")
    
    if not source_base.exists():
        print(f"Error: Source directory {source_base} does not exist.")
        return

    # 2. Use rglob to recursively find all PET.nii.gz files, no matter how deep
    for pet_path in source_base.rglob('PET.nii.gz'):
        # The folder containing the PET file is our working scan directory
        scan_dir = pet_path.parent
        
        # Look for the CT file in the same directory
        ct_path = scan_dir / 'CT_resample.nii.gz'
        if not ct_path.exists():
            ct_path = scan_dir / 'CT.nii.gz'

        if ct_path.exists():
            try:
                # Maintain original nested folder structure for saving
                relative_dir = scan_dir.relative_to(source_base)
                save_path = target_base / relative_dir / 'PET.npy'
                
                print(f"Processing scan: {relative_dir}")
                
                # ---------------- Load Data & Extract Metadata ----------------
                pet_img = sitk.ReadImage(str(pet_path))
                ct_img = sitk.ReadImage(str(ct_path))
                
                spacing_xyz = pet_img.GetSpacing()
                spacing_xyz_rounded = np.round(spacing_xyz).astype(float)
                
                pet_array = sitk.GetArrayFromImage(pet_img).astype(np.float32)
                ct_array = sitk.GetArrayFromImage(ct_img).astype(np.float32)
                
                orig_spacing_zyx = np.array([
                    spacing_xyz_rounded[2], 
                    spacing_xyz_rounded[1], 
                    spacing_xyz_rounded[0]
                ])
                
                # ---------------- Resampling ----------------
                zoom_factors_pet = orig_spacing_zyx / target_spacing_zyx
                pet_resampled = scipy.ndimage.zoom(pet_array, zoom_factors_pet, order=1)
                
                zoom_factors_ct = np.array(pet_resampled.shape) / np.array(ct_array.shape)
                ct_resampled = scipy.ndimage.zoom(ct_array, zoom_factors_ct, order=1)
                
                # ---------------- Cropping ----------------
                bbox = get_bounding_box(pet_resampled, ct_resampled)
                if bbox is not None:
                    pet_cropped = pet_resampled[bbox]
                    ct_cropped = ct_resampled[bbox]
                else:
                    pet_cropped, ct_cropped = pet_resampled, ct_resampled
                    
                # ---------------- Normalization & Stacking ----------------
                pet_norm = normalize_volume(pet_cropped, method='z-score')
                ct_norm = normalize_volume(ct_cropped, method='z-score')
                
                stacked_volume = np.stack([pet_norm, ct_norm], axis=0)
                
                # ---------------- Saving ----------------
                save_path.parent.mkdir(parents=True, exist_ok=True)
                np.save(save_path, stacked_volume)
                
                print(f"  -> Extracted Spacing (XYZ): {spacing_xyz} -> Rounded: {spacing_xyz_rounded}")
                print(f"  -> Saved to: {save_path}")
                print(f"  -> Final Tensor Shape: {stacked_volume.shape}\n")
                
            except Exception as e:
                print(f"  -> [Error] Failed processing {scan_dir.name}: {e}\n")

if __name__ == '__main__':
    process_autopet_dataset()
    print("AutoPET dataset processing complete.")